# Exercise 2
Today we are going to continue to work on point clouds.
We will work on clustering point clouds. That enables us to segment them.

In [178]:
import numpy as np
import open3d as o3d
import copy
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.cluster import KMeans, k_means

In [179]:
def draw_labels_on_model(pcl,labels):
    cmap = plt.get_cmap("tab20")
    pcl_temp = copy.deepcopy(pcl)
    max_label = labels.max()
    print("%s has %d clusters" % (pcl_name, max_label + 1))
    colors = cmap(labels / (max_label if max_label > 0 else 1))
    colors[labels < 0] = 0
    pcl_temp.colors = o3d.utility.Vector3dVector(colors[:, :3])
    o3d.visualization.draw_geometries([pcl_temp])



## K-means on a cube
We created a point cloud using `open3d`.
Our goal is to segment each side using k-means.

In [180]:
pcl_name = 'Cube'
density = 1e4 # density of sample points to create
pcl = o3d.geometry.TriangleMesh.create_box().sample_points_uniformly(int(density))
eps = 0.4
print("%s has %d points" % (pcl_name, np.asarray(pcl.points).shape[0]))
o3d.visualization.draw_geometries([pcl])

Cube has 10000 points


If we just use k-means out of the box with the point cloud, we will get what just has been visualized.

Note: Using the '+' and '-' keys in the viewer will increase/decrease the size of the points.

In [181]:
km = KMeans(n_clusters=6, init='random',
            n_init=10, max_iter=300, tol=1e-04, random_state=0)

# Get the points from the pointcloud as nparray
xyz = np.asarray(pcl.points)
labels = km.fit_predict(xyz)
draw_labels_on_model(pcl, labels)

Cube has 6 clusters


We can see that we get six clusters, but they do not span a side.

We try again, but this time we instead use the normals of the cube as input for k-means.

The normals for each plane should be parallel with the other normals from said plane.

In [182]:

pcl = o3d.geometry.TriangleMesh.create_box().sample_points_uniformly(int(1e4), use_triangle_normal=True)

normals = np.asarray(pcl.normals)
km_normals = KMeans(n_clusters=6, init='random', n_init=10, max_iter=300, tol=1e-04, random_state=0)
labels_normals = km_normals.fit_predict(normals)
draw_labels_on_model(pcl, labels_normals)

Cube has 6 clusters


This still does not work, opposite sides will also have normals that point the other way ($\vec{n}$ and $-\vec{n}$).

So, to combat this we can attempt to use the xyz coordinates and the normals.

## More exercises

### A) K-means continued.

Combine the point cloud points (xyz) with the normals and do k-means.

```xyz_n = np.concatenate((xyz, normals), axis=1)```

Do you get better clusters?
Why would adding the normals help?

### B) 
Try weighting either the points or normals by scaling them by some factor. Can this perfectly segment each of the faces of the cube?
### C)
Try to cluster all the different shapes using k means.
```{Python}
d = 4
mesh = o3d.geometry.TriangleMesh.create_tetrahedron().translate((-d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_octahedron().translate((0, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_icosahedron().translate((d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_torus().translate((-d, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_moebius(twists=1).translate(
    (0, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_moebius(twists=2).translate(
    (d, -d, 0))
mesh.sample_points_uniformly(int(1e5)), 0.5
```

### D)
Now try segmenting a different point cloud located at `pointclouds/fragment.ply`
Are you able to cluster the point cloud?

Which features could be useful to segment this point cloud?
- fpfh features?
- xyz
- normals 
- colors

Are you able to get clusters that make sense? Why?

### E)
Use the built-in `cluster_dbscan` algorithm.
Tweak the parameters and see what you get out.

Attempt on the combined figures and on `fragment.ply`
```{Python}
#eps (float) – Density parameter that is used to find neighbouring points.
eps = 0.02

#min_points (int) – Minimum number of points to form a cluster.
min_points = 10

labels = np.array(pcl.cluster_dbscan(eps=eps, min_points=min_points, print_progress=True))
```


In [183]:
d = 4
mesh = o3d.geometry.TriangleMesh.create_tetrahedron().translate((-d, 0, 0))
# Combine xyz coordinates with normals
xyz_n = np.concatenate((xyz, normals), axis=1)

# Apply K-means clustering on combined xyz + normals
km_xyzn = KMeans(n_clusters=6, init='random',
                 n_init=10, max_iter=300, tol=1e-04, random_state=0)
labels_xyzn = km_xyzn.fit_predict(xyz_n)

# Visualize the result
draw_labels_on_model(pcl, labels_xyzn)


Cube has 6 clusters


In [184]:
# --- B) Weighting to get clean 6 faces ---

# Combine xyz coordinates with normals and apply weighting
w_xyz = 3.0       # weight for point coordinates
w_normals = 1.0   # weight for normals

xyz_n = np.concatenate((xyz * w_xyz, np.abs(normals) * w_normals), axis=1)  # abs(normals) handles opposite directions

# Apply K-means clustering on weighted xyz + normals
km_xyzn = KMeans(n_clusters=6, init='k-means++',
                 n_init=50, max_iter=300, tol=1e-04, random_state=0)
labels_xyzn = km_xyzn.fit_predict(xyz_n)

# Visualize the result
draw_labels_on_model(pcl, labels_xyzn)


Cube has 6 clusters


In [185]:
# --- C) Clustering multiple 3D shapes using K-means (Open3D stable version) ---

d = 4  # distance between shapes

# Create and position multiple shapes
mesh = o3d.geometry.TriangleMesh.create_tetrahedron().translate((-d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_octahedron().translate((0, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_icosahedron().translate((d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_torus().translate((-d, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_sphere().translate((0, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_cone().translate((d, -d, 0))

# Sample points with normals
pcl = mesh.sample_points_uniformly(int(1e5), use_triangle_normal=True)

# Extract coordinates and normals
xyz = np.asarray(pcl.points)
normals = np.asarray(pcl.normals)

print("Point cloud contains:", xyz.shape[0], "points")

# Combine xyz + normals
xyz_n = np.concatenate((xyz, np.abs(normals)), axis=1)

# Apply K-means clustering
km_shapes = KMeans(n_clusters=8, init='k-means++',
                   n_init=30, max_iter=300, tol=1e-04, random_state=0)
labels_shapes = km_shapes.fit_predict(xyz_n)

# Visualize the clusters
draw_labels_on_model(pcl, labels_shapes)


Point cloud contains: 100000 points
Cube has 8 clusters


In [187]:
# --- D) Cluster fragment.ply ---
import open3d as o3d, numpy as np
from sklearn.cluster import KMeans

# Load and check file
frag = o3d.io.read_point_cloud("TestData/fragment.ply")
if len(frag.points) == 0:
    raise Exception("⚠️ fragment.ply not found or empty. Check path!")

# Estimate normals
frag.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=50))

xyz = np.asarray(frag.points)
normals = np.asarray(frag.normals)
colors = np.asarray(frag.colors) if frag.has_colors() else np.zeros_like(xyz)

# Try XYZ + normals + colors
X = np.concatenate([xyz, np.abs(normals), colors], axis=1)

# KMeans
km = KMeans(n_clusters=8, n_init=20, random_state=0)
labels = km.fit_predict(X)

draw_labels_on_model(frag, labels)


Cube has 8 clusters


In [188]:
# --- E) DBSCAN clustering on fragment.ply ---
import numpy as np
import open3d as o3d

# Load the fragment file
frag = o3d.io.read_point_cloud("TestData/fragment.ply")

# Estimate normals for better clustering
frag.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=50))

# DBSCAN parameters
eps = 0.02          # neighborhood radius
min_points = 10     # minimum points to form a cluster

# Run DBSCAN
labels = np.array(frag.cluster_dbscan(eps=eps, min_points=min_points, print_progress=True))

# Show clusters
print(f"DBSCAN found {labels.max() + 1} clusters.")
draw_labels_on_model(frag, labels)


DBSCAN found 10 clusters.
Cube has 10 clusters
